In [ ]:
%%capture
!pip install arch pmdarima stable_baselines3

In [ ]:
# Setup and training data

import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import yfinance as yf
from arch import arch_model
from pmdarima import auto_arima
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import SAC
from stable_baselines3.common.env_checker import check_env
TRAIN_START = "2022-01-01"
TRAIN_END   = "2025-05-31"
REBALANCE_FREQUENCY = "weekly"
TRANSACTION_COST_RATE = 0.001
BENCHMARK_TICKERS = ["^CRSLDX", "SETF10GILT.NS"]
BENCHMARK_WEIGHTS = np.array([0.60, 0.40])
BENCHMARK_NAME = "60/40 Nifty TRI-Gilt Benchmark"

stocks = [
    "GOLDBEES.NS", "SILVERBEES.NS",
    "SETF10GILT.NS",
    "INDIGRID.NS",
    "MCX.NS",
    "LT.NS",
    "NTPC.NS",
    "COALINDIA.NS",
    "PIDILITIND.NS",
    "WELCORP.NS",
    "CUMMINSIND.NS",
    "MANORAMA.NS",
    "SOLARINDS",
]
print("Downloading market data...")
price_train = yf.download(stocks, start=TRAIN_START, end=TRAIN_END, progress=False)["Close"]
price_train = price_train.ffill()
missing_threshold = int(len(price_train) * 0.05)
price_train = price_train.dropna(axis=1, thresh=len(price_train) - missing_threshold)
price_train = price_train.dropna()
stocks = list(price_train.columns)
log_returns_train = np.log(price_train / price_train.shift(1)).dropna()

print(f"Successfully loaded {len(stocks)} assets.")
print(f"Training shape: {log_returns_train.shape}")

In [ ]:
# Correlation check

import plotly.express as px
corr_matrix = log_returns_train.corr()
fig = px.imshow(
    corr_matrix,
    text_auto=".2f",
    aspect="auto",
    color_continuous_scale='RdBu_r',
    title="Asset Correlation Heatmap",
    labels=dict(x="Assets", y="Assets", color="Correlation")
)
fig.update_layout(
    xaxis_title="",
    yaxis_title="",
    width=800,
    height=800,
    coloraxis_colorbar=dict(title="Coefficient")
)
fig.show()

In [ ]:
# Build features

print("Building rolling features...")
rolling_returns = log_returns_train.rolling(window=21).mean().fillna(0)
rolling_vol = log_returns_train.rolling(window=21).std().fillna(0)
garch_volatility = pd.DataFrame(index=log_returns_train.index, columns=stocks)

for asset in stocks:
    am = arch_model(log_returns_train[asset] * 100, vol='Garch', p=1, q=1, dist='Normal')
    res = am.fit(disp='off')
    garch_volatility[asset] = res.conditional_volatility / 100
feat_1 = rolling_returns.add_suffix('_roll_ret')
feat_2 = rolling_vol.add_suffix('_roll_vol')
feat_3 = garch_volatility.add_suffix('_garch_vol')
features_train = pd.concat([feat_1, feat_2, feat_3], axis=1)
features_train = features_train.dropna()
log_returns_train = log_returns_train.loc[features_train.index]

print(f"Feature matrix shape: {features_train.shape}")
print("Features ready.")

In [ ]:
# Portfolio environment

from scipy.special import softmax
import numpy as np
import pandas as pd
import gymnasium as gym
from gymnasium import spaces

class PortfolioEnv(gym.Env):
    metadata = {'render_modes': ['console']}

    def __init__(self, df_features, df_returns, initial_capital=100000.0, rebalance_frequency=REBALANCE_FREQUENCY, transaction_cost_rate=0.001):
        super(PortfolioEnv, self).__init__()

        self.df_features = df_features
        self.df_returns = df_returns
        self.assets = df_returns.columns.tolist()
        self.num_assets = len(self.assets)

        self.initial_capital = initial_capital
        self.transaction_cost_rate = transaction_cost_rate
        frequency_aliases = {
            "half_yearly": "semiannual", "halfyearly": "semiannual",
            "semi_annually": "semiannual", "yearly": "annual", "annually": "annual"
        }
        requested_frequency = str(rebalance_frequency).lower().replace("-", "_").replace(" ", "_")
        self.rebalance_frequency = frequency_aliases.get(requested_frequency, requested_frequency)
        rebalance_intervals = {
            "daily": pd.DateOffset(days=1),
            "weekly": pd.DateOffset(weeks=1),
            "fortnightly": pd.DateOffset(weeks=2),
            "monthly": pd.DateOffset(months=1),
            "quarterly": pd.DateOffset(months=3),
            "semiannual": pd.DateOffset(months=6),
            "annual": pd.DateOffset(years=1),
        }
        if self.rebalance_frequency not in rebalance_intervals:
            valid_frequencies = ", ".join(rebalance_intervals)
            raise ValueError(f"Unsupported rebalance_frequency: {rebalance_frequency}. Choose one of: {valid_frequencies}.")
        self.rebalance_interval = rebalance_intervals[self.rebalance_frequency]
        self.max_cap = 0.05
        target_capped_assets = ["GOLDBEES.NS", "SILVERBEES.NS", "EMBASSY.NS", "INDIGRID.NS"]
        self.capped_indices = [self.assets.index(asset) for asset in target_capped_assets if asset in self.assets]
        self.action_space = spaces.Box(low=-1.0, high=1.0, shape=(self.num_assets,), dtype=np.float32)
        self.obs_dim = self.df_features.shape[1] + self.num_assets + 1
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(self.obs_dim,), dtype=np.float32)

        self.current_step = 0
        self.capital = self.initial_capital
        self.portfolio_weights = np.ones(self.num_assets, dtype=np.float32) / self.num_assets
        self.next_rebalance_date = pd.Timestamp(self.df_returns.index[0])

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.current_step = 0
        self.capital = self.initial_capital
        self.portfolio_weights = np.ones(self.num_assets, dtype=np.float32) / self.num_assets
        self.next_rebalance_date = pd.Timestamp(self.df_returns.index[0])
        return self._get_obs(), {}

    def _should_rebalance(self):
        current_date = pd.Timestamp(self.df_returns.index[self.current_step])
        return current_date >= self.next_rebalance_date

    def _get_regime(self):
        if self.current_step < 21:
            return 0.0
        recent_returns = self.df_returns.iloc[self.current_step-21 : self.current_step]
        market_momentum = np.mean(np.sum(recent_returns.values, axis=0))
        if market_momentum > 0.02: return 1.0
        elif market_momentum < -0.02: return -1.0
        else: return 0.0

    def _get_obs(self):
        market_features = self.df_features.iloc[self.current_step].values.astype(np.float32)
        regime_flag = np.array([self._get_regime()], dtype=np.float32)
        return np.concatenate([market_features, self.portfolio_weights, regime_flag])

    def step(self, action):
        rebalance_today = self._should_rebalance()
        new_weights = self.portfolio_weights.copy()

        if rebalance_today:
            new_weights = softmax(action)
            excess_weight = 0.0
            for idx in self.capped_indices:
                if new_weights[idx] > self.max_cap:
                    excess_weight += (new_weights[idx] - self.max_cap)
                    new_weights[idx] = self.max_cap
            if excess_weight > 0:
                mask = np.ones(self.num_assets, dtype=bool)
                for idx in self.capped_indices:
                    mask[idx] = False
                remaining_sum = np.sum(new_weights[mask])
                if remaining_sum > 0:
                    new_weights[mask] += (new_weights[mask] / remaining_sum) * excess_weight

            current_date = pd.Timestamp(self.df_returns.index[self.current_step])
            self.next_rebalance_date = current_date + self.rebalance_interval

        turnover = np.abs(new_weights - self.portfolio_weights).sum() if rebalance_today else 0.0
        cost_impact = turnover * self.transaction_cost_rate

        asset_returns = np.exp(self.df_returns.iloc[self.current_step].values) - 1
        portfolio_gross_return = np.dot(new_weights, asset_returns)
        portfolio_net_return = portfolio_gross_return - cost_impact
        benchmark_return = np.mean(asset_returns)
        excess_return = portfolio_net_return - benchmark_return

        downside_loss = max(0.0, -portfolio_net_return)
        downside_penalty = 5.0 * (downside_loss ** 2)
        reward = (excess_return - downside_penalty) * 100.0

        previous_capital = self.capital
        self.capital = self.capital * (1 + portfolio_net_return)

        self.portfolio_weights = new_weights
        self.current_step += 1
        terminated = self.current_step >= len(self.df_returns) - 1
        truncated = False

        info = {
            'portfolio_value': self.capital,
            'portfolio_return': portfolio_net_return,
            'portfolio_gross_return': portfolio_gross_return,
            'transaction_cost': cost_impact,
            'turnover': turnover,
            'benchmark_return': benchmark_return,
            'excess_return': excess_return,
            'rebalanced': rebalance_today,
            'rebalance_frequency': self.rebalance_frequency,
            'regime': self._get_regime()
        }

        next_obs = self._get_obs() if not terminated else np.zeros(self.obs_dim, dtype=np.float32)

        return next_obs, float(reward), terminated, truncated, info

In [ ]:
# Train SAC

import torch
from stable_baselines3 import SAC
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3.common.logger import configure
def make_env():
    return PortfolioEnv(features_train, log_returns_train, rebalance_frequency=REBALANCE_FREQUENCY, transaction_cost_rate=TRANSACTION_COST_RATE)

env = DummyVecEnv([make_env])
env = VecNormalize(env, norm_obs=True, norm_reward=False, clip_obs=10.)
policy_kwargs = dict(net_arch=dict(pi=[256, 256], qf=[256, 256]))

model = SAC(
    "MlpPolicy",
    env,
    verbose=1,
    learning_rate=3e-4,
    batch_size=256,
    buffer_size=100000,
    ent_coef='auto',
    policy_kwargs=policy_kwargs,
    device="cuda",
    seed=42
)
log_path = "./sac_training_logs/"
new_logger = configure(log_path, ["stdout", "csv"])
model.set_logger(new_logger)
print("Training model...")
model.learn(total_timesteps=100000, progress_bar=True)
env.save("vec_normalize.pkl")
model.save("sac_portfolio_allocator")

print("Training finished.")

In [ ]:
# Training loss

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

print("Plotting actor and critic loss...")
log_df = pd.read_csv("./sac_training_logs/progress.csv")
actor_loss = log_df[['time/total_timesteps', 'train/actor_loss']].dropna()
critic_loss = log_df[['time/total_timesteps', 'train/critic_loss']].dropna()
fig_loss = make_subplots(specs=[[{"secondary_y": True}]])
fig_loss.add_trace(
    go.Scatter(
        x=critic_loss['time/total_timesteps'],
        y=critic_loss['train/critic_loss'],
        name="Critic Loss (MSE)",
        mode="lines",
        line=dict(color="#d62728", width=2)
    ),
    secondary_y=False,
)
fig_loss.add_trace(
    go.Scatter(
        x=actor_loss['time/total_timesteps'],
        y=actor_loss['train/actor_loss'],
        name="Actor Loss",
        mode="lines",
        line=dict(color="#1f77b4", width=2)
    ),
    secondary_y=True,
)
fig_loss.update_layout(
    title_text="SAC training loss",
    xaxis_title="Total Timesteps",
    template="plotly_white",
    hovermode="x unified",
    legend=dict(yanchor="top", y=0.99, xanchor="right", x=0.99, bordercolor="gray", borderwidth=1)
)

fig_loss.update_yaxes(title_text="Critic Loss", secondary_y=False, title_font=dict(color="#d62728"))
fig_loss.update_yaxes(title_text="Actor Loss", secondary_y=True, title_font=dict(color="#1f77b4"))

fig_loss.show()

In [ ]:
# Test data

print("Preparing test data...")

BACKTEST_START = "2025-06-01"
BACKTEST_END   = "2026-06-25"
FETCH_START    = "2025-05-01"
price_test = yf.download(stocks, start=FETCH_START, end=BACKTEST_END, progress=False)["Close"]
price_test = price_test.ffill().dropna()

log_returns_test_full = np.log(price_test / price_test.shift(1)).dropna()
rolling_returns_test = log_returns_test_full.rolling(window=21).mean()
rolling_vol_test = log_returns_test_full.rolling(window=21).std()
garch_volatility_test = pd.DataFrame(index=log_returns_test_full.index, columns=stocks)
for asset in stocks:
    am = arch_model(log_returns_test_full[asset] * 100, vol='Garch', p=1, q=1, dist='Normal')
    res = am.fit(disp='off')
    garch_volatility_test[asset] = res.conditional_volatility / 100
feat_1_t = rolling_returns_test.add_suffix('_roll_ret')
feat_2_t = rolling_vol_test.add_suffix('_roll_vol')
feat_3_t = garch_volatility_test.add_suffix('_garch_vol')

features_test_full = pd.concat([feat_1_t, feat_2_t, feat_3_t], axis=1).dropna()
features_test = features_test_full.loc[BACKTEST_START:BACKTEST_END]
log_returns_test = log_returns_test_full.loc[features_test.index]

print(f"Test feature matrix shape: {features_test.shape}")

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px

# Backtest
print("Running backtest...")
eval_env = PortfolioEnv(features_test, log_returns_test, initial_capital=1.0, rebalance_frequency=REBALANCE_FREQUENCY, transaction_cost_rate=TRANSACTION_COST_RATE)
model = SAC.load("sac_portfolio_allocator")
drl_weights_history = []
drl_returns_history = []

obs, _ = eval_env.reset()
terminated = False

while not terminated:
    action, _states = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = eval_env.step(action)
    drl_weights_history.append(eval_env.portfolio_weights)
    drl_returns_history.append(info['portfolio_return'])

eval_dates = log_returns_test.index[:-1]
drl_weights_df = pd.DataFrame(
    drl_weights_history,
    columns=stocks,
    index=eval_dates
)
drl_daily_returns = pd.Series(
    drl_returns_history,
    index=eval_dates,
    name="DRL_Portfolio"
)
drl_portfolio_cum = np.exp(drl_daily_returns.cumsum())

print("Backtest finished. Recent weights:")
display(drl_weights_df.tail())

# Portfolio weights over time
fig_weights = px.area(
    drl_weights_df,
    x=drl_weights_df.index,
    y=drl_weights_df.columns,
    title="Portfolio weights over time",
    labels={"value": "Weight Allocation"}
)

fig_weights.update_layout(
    template="plotly_white",
    hovermode="x unified",
    yaxis=dict(
        title="Portfolio Weight",
        range=[0, 1],
        tickformat=".0%"  # Displays axis as percentages (0% - 100%)
    ),
    xaxis=dict(title="Date"),
    legend_title_text="Assets",
    height=550,
    margin=dict(l=40, r=40, t=60, b=40)
)

fig_weights.show()

In [ ]:
# Performance metrics

RISK_FREE_RATE = 0.06716

def perf_stats(returns):
    ann_ret = (np.exp(returns.mean() * 252) - 1)
    ann_vol = returns.std() * np.sqrt(252)
    sharpe = (ann_ret - RISK_FREE_RATE) / ann_vol
    down = returns[returns < 0]
    sortino = (ann_ret - RISK_FREE_RATE) / (down.std() * np.sqrt(252))

    cumulative = np.exp(returns.cumsum())
    dd = cumulative.cummax()
    drawdown = (cumulative - dd) / dd
    max_dd = drawdown.min()

    calmar = ann_ret / abs(max_dd) if max_dd != 0 else np.nan

    return pd.Series({
        "Annual Return": ann_ret,
        "Annual Volatility": ann_vol,
        "Sharpe Ratio": sharpe,
        "Sortino Ratio": sortino,
        "Max Drawdown": max_dd,
        "Calmar Ratio": calmar
    })

print(f"Fetching {BENCHMARK_NAME} data...")

bench_data = yf.download(BENCHMARK_TICKERS, start=BACKTEST_START, end=BACKTEST_END, progress=False)["Close"]
bench_data = bench_data.ffill().dropna()

bench_pct_returns = bench_data.pct_change().dropna()
bench_simple_portfolio_ret = (bench_pct_returns * BENCHMARK_WEIGHTS).sum(axis=1)
bench_daily_ret = np.log(1 + bench_simple_portfolio_ret)
num_assets = len(stocks)
equi_weights = np.ones(num_assets) / num_assets
max_cap = 0.05
target_capped_assets = ["GOLDBEES.NS", "SILVERBEES.NS"]
capped_indices = [i for i, s in enumerate(stocks) if s in target_capped_assets]
if (1.0 / num_assets) > max_cap and len(capped_indices) > 0:
    for idx in capped_indices:
        equi_weights[idx] = max_cap

    remaining_weight = 1.0 - (len(capped_indices) * max_cap)
    non_capped_count = num_assets - len(capped_indices)

    for i in range(num_assets):
        if i not in capped_indices:
            equi_weights[i] = remaining_weight / non_capped_count

simple_returns_test = np.exp(log_returns_test) - 1
equi_simple_ret = (simple_returns_test * equi_weights).sum(axis=1)
equi_daily_ret = np.log(1 + equi_simple_ret)
common_dates = drl_daily_returns.index.intersection(bench_daily_ret.index).intersection(equi_daily_ret.index)
drl_aligned = drl_daily_returns.loc[common_dates]
bench_aligned = bench_daily_ret.loc[common_dates]
equi_aligned = equi_daily_ret.loc[common_dates]

drl_stats = perf_stats(drl_aligned)
bench_stats = perf_stats(bench_aligned)
equi_stats = perf_stats(equi_aligned)

perf_df = pd.concat([drl_stats, bench_stats, equi_stats], axis=1)
perf_df.columns = ["RL Agent", BENCHMARK_NAME, "Capped Equiweighted"]

print("\nOut-of-sample performance:")
display(perf_df)

In [ ]:
# Backtest performance

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

print("Plotting performance...")

fig_backtest = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.05,
    row_heights=[0.7, 0.3]
)
fig_backtest.add_trace(go.Scatter(
    x=common_dates,
    y=np.exp(drl_aligned.cumsum()),
    mode='lines+markers',
    name='RL Agent',
    line=dict(color='#006400', width=2.5),
    marker=dict(size=4)
), row=1, col=1)
fig_backtest.add_trace(go.Scatter(
    x=common_dates,
    y=np.exp(bench_aligned.cumsum()),
    mode='lines',
    name=BENCHMARK_NAME,
    line=dict(color='#7f7f7f', width=2, dash='dash')
), row=1, col=1)
fig_backtest.add_trace(go.Scatter(
    x=common_dates,
    y=np.exp(equi_aligned.cumsum()),
    mode='lines',
    name='Equiweighted (1/N)',
    line=dict(color='#ff7f0e', width=2, dash='dot')
), row=1, col=1)
fig_backtest.add_trace(go.Scatter(
    x=common_dates,
    y=bench_aligned,
    mode='lines',
    name=f'{BENCHMARK_NAME} Daily Return',
    line=dict(color='#1f77b4', width=1)
), row=2, col=1)

fig_backtest.update_layout(
    title='Out-of-Sample Portfolio Growth vs. Benchmarks',
    xaxis_title='Date',
    template='plotly_white',
    hovermode='x unified',
    height=750,
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01, bordercolor="gray", borderwidth=1)
)
fig_backtest.update_yaxes(title_text="Growth of ₹1", row=1, col=1)
fig_backtest.update_yaxes(title_text="Daily Return", tickformat=".1%", row=2, col=1)

fig_backtest.show()

In [ ]:
# Volatility regimes

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

print("Checking volatility regimes...")
rolling_vol_bench = bench_aligned.rolling(window=21).std() * np.sqrt(252)
high_vol_thresh = rolling_vol_bench.quantile(0.75)
low_vol_thresh = rolling_vol_bench.quantile(0.25)
fig_vol = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.05,
    row_heights=[0.7, 0.3]
)
fig_vol.add_trace(go.Scatter(
    x=drl_portfolio_cum.index, y=drl_portfolio_cum,
    mode='lines', name='RL Agent',
    line=dict(color='#2ca02c', width=2.5)
), row=1, col=1)

fig_vol.add_trace(go.Scatter(
    x=bench_aligned.index, y=np.exp(bench_aligned.cumsum()),
    mode='lines', name=BENCHMARK_NAME,
    line=dict(color='#7f7f7f', width=2, dash='dash')
), row=1, col=1)
fig_vol.add_trace(go.Scatter(
    x=rolling_vol_bench.index, y=rolling_vol_bench,
    mode='lines', name='Benchmark 21d Volatility',
    line=dict(color='#1f77b4', width=1.5)
), row=2, col=1)
fig_vol.add_hrect(
    y0=high_vol_thresh, y1=rolling_vol_bench.max() * 1.1,
    fillcolor="red", opacity=0.1, line_width=0,
    annotation_text="High Volatility Zone", annotation_position="top left",
    row=2, col=1
)
fig_vol.add_hrect(
    y0=0, y1=low_vol_thresh,
    fillcolor="green", opacity=0.1, line_width=0,
    annotation_text="Low Volatility Zone", annotation_position="bottom left",
    row=2, col=1
)
fig_vol.add_hline(y=high_vol_thresh, line_dash="dot", line_color="red", row=2, col=1)
fig_vol.add_hline(y=low_vol_thresh, line_dash="dot", line_color="green", row=2, col=1)
fig_vol.update_layout(
    title='Performance across volatility regimes',
    template='plotly_white',
    hovermode='x unified',
    height=750,
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01, bordercolor="gray", borderwidth=1)
)

fig_vol.update_yaxes(title_text="Growth of ₹1", row=1, col=1)
fig_vol.update_yaxes(title_text="Ann. Volatility", tickformat=".1%", row=2, col=1)

fig_vol.show()

In [ ]:
# Turnover

import plotly.graph_objects as go
import numpy as np

print("Checking turnover...")
daily_turnover = drl_weights_df.diff().abs().sum(axis=1).fillna(0)
transaction_threshold = 0.01
transaction_days = daily_turnover[daily_turnover > transaction_threshold]
num_transactions = len(transaction_days)
annual_turnover = daily_turnover.mean() * 252

print(f"Total Trading Days in Backtest: {len(daily_turnover)}")
print(f"Number of Meaningful Transactions (>1% Turnover): {num_transactions}")
print(f"Average Daily Turnover: {daily_turnover.mean():.2%}")
print(f"Annualized Portfolio Turnover: {annual_turnover:.2%}")
fig_turnover = go.Figure()
fig_turnover.add_trace(go.Bar(
    x=daily_turnover.index,
    y=daily_turnover,
    name='Daily Portfolio Turnover',
    marker_color='#1f77b4',
    opacity=0.8
))
fig_turnover.add_hline(
    y=transaction_threshold,
    line_dash="dot",
    line_color="red",
    annotation_text="1% Transaction Threshold",
    annotation_position="top right"
)

fig_turnover.update_layout(
    title=f'Agent Transaction Frequency (Total Meaningful Trades: {num_transactions})',
    xaxis_title='Date',
    yaxis_title='Turnover Magnitude',
    yaxis_tickformat='.1%',
    template='plotly_white',
    hovermode='x unified',
    height=500
)

fig_turnover.show()

In [ ]:
# Risk metrics

print("Calculating risk metrics...")
covariance_matrix = np.cov(drl_aligned, bench_aligned)
portfolio_beta = covariance_matrix[0, 1] / covariance_matrix[1, 1]
ann_ret_agent = perf_df.loc["Annual Return", "RL Agent"]
ann_ret_bench = perf_df.loc["Annual Return", BENCHMARK_NAME]
jensens_alpha = ann_ret_agent - (RISK_FREE_RATE + portfolio_beta * (ann_ret_bench - RISK_FREE_RATE))
arithmetic_alpha = ann_ret_agent - ann_ret_bench
daily_active_returns = drl_aligned - bench_aligned
tracking_error = daily_active_returns.std() * np.sqrt(252)
information_ratio = arithmetic_alpha / tracking_error if tracking_error != 0 else np.nan
advanced_stats = pd.Series({
    "Portfolio Beta": portfolio_beta,
    "Arithmetic Alpha": arithmetic_alpha,
    "Jensen's Alpha": jensens_alpha,
    "Tracking Error": tracking_error,
    "Information Ratio": information_ratio
})

print("\n--- Risk metrics ---")
for metric, val in advanced_stats.items():
    if "Ratio" in metric or "Beta" in metric:
        print(f"{metric:<20}: {val:.3f}")
    else:
        print(f"{metric:<20}: {val:.2%}")

In [ ]:
# Risk/return data

print("Calculating asset risk and return...")
asset_ann_returns = (np.exp(log_returns_test.mean() * 252) - 1) * 100
asset_ann_volatility = (log_returns_test.std() * np.sqrt(252)) * 100
scatter_df = pd.DataFrame({
    'Risk (Volatility %)': asset_ann_volatility,
    'Return (%)': asset_ann_returns,
    'Type': 'Individual Asset'
})
drl_ann_ret = (np.exp(drl_aligned.mean() * 252) - 1) * 100
drl_ann_vol = (drl_aligned.std() * np.sqrt(252)) * 100
bench_ann_ret = (np.exp(bench_aligned.mean() * 252) - 1) * 100
bench_ann_vol = (bench_aligned.std() * np.sqrt(252)) * 100

In [ ]:
# Risk/return comparison

import plotly.graph_objects as go

fig_scatter = go.Figure()
fig_scatter.add_trace(go.Scatter(
    x=scatter_df['Risk (Volatility %)'],
    y=scatter_df['Return (%)'],
    mode='markers+text',
    text=scatter_df.index,
    textposition="top center",
    name='Underlying Assets',
    marker=dict(color='#e74c3c', size=12, line=dict(color='white', width=1)),
    hovertemplate='<b>%{text}</b><br>Volatility: %{x:.2f}%<br>Return: %{y:.2f}%<extra></extra>'
))
fig_scatter.add_trace(go.Scatter(
    x=[drl_ann_vol],
    y=[drl_ann_ret],
    mode='markers+text',
    text=['DEEP RL AGENT'],
    textposition="bottom center",
    name='RL Strategy',
    marker=dict(color='#110487', size=18, symbol='diamond', line=dict(color='white', width=1.5)),
    hovertemplate='<b>%{text}</b><br>Volatility: %{x:.2f}%<br>Return: %{y:.2f}%<extra></extra>'
))
fig_scatter.add_trace(go.Scatter(
    x=[bench_ann_vol],
    y=[bench_ann_ret],
    mode='markers+text',
    text=['BENCHMARK'],
    textposition="bottom center",
    name='Benchmark',
    marker=dict(color='#7f7f7f', size=16, symbol='square', line=dict(color='white', width=1.5)),
    hovertemplate='<b>%{text}</b><br>Volatility: %{x:.2f}%<br>Return: %{y:.2f}%<extra></extra>'
))
fig_scatter.update_layout(
    title=dict(text='Risk vs. return', x=0.5, font=dict(size=18)),
    xaxis_title='Annualized Volatility (Risk %)',
    yaxis_title='Expected Annual Return (%)',
    template='plotly_white',
    height=650,
    hovermode='closest',
    grid=dict(rows=1, columns=1),
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01, bordercolor="gray", borderwidth=1)
)
fig_scatter.show()

In [ ]:
# Strategy/benchmark correlation

import plotly.graph_objects as go

print("Calculating strategy/benchmark correlation...")
strategy_corr = drl_aligned.corr(bench_aligned)
print(f"Pearson correlation (agent vs. benchmark): {strategy_corr:.3f}")
rolling_corr = drl_aligned.rolling(window=60).corr(bench_aligned).dropna()
fig_roll_corr = go.Figure()

fig_roll_corr.add_trace(go.Scatter(
    x=rolling_corr.index,
    y=rolling_corr,
    mode='lines',
    name='60-Day Rolling Correlation',
    line=dict(color='#8c564b', width=2),
    fill='tozeroy',
    fillcolor='rgba(140, 86, 75, 0.2)'
))
fig_roll_corr.add_hline(y=0, line_dash="dash", line_color="black", annotation_text="Zero Correlation")

fig_roll_corr.update_layout(
    title=f"60-Day Rolling Correlation: RL Agent vs. {BENCHMARK_NAME}",
    xaxis_title="Date",
    yaxis_title="Pearson Correlation (r)",
    template="plotly_white",
    hovermode="x unified",
    yaxis=dict(range=[-1.1, 1.1])
)

fig_roll_corr.show()

In [ ]:
# Forward test data

print("Preparing forward-test data...")
FORWARD_START = "2026-06-26"
FORWARD_END   = "2026-09-4"
FETCH_START_FORWARD = "2026-05-20"
price_forward = yf.download(stocks, start=FETCH_START_FORWARD, end=FORWARD_END, progress=False)["Close"]
price_forward = price_forward.ffill().dropna()

log_returns_forward_full = np.log(price_forward / price_forward.shift(1)).dropna()
rolling_returns_forward = log_returns_forward_full.rolling(window=21).mean()
rolling_vol_forward = log_returns_forward_full.rolling(window=21).std()
garch_volatility_forward = pd.DataFrame(index=log_returns_forward_full.index, columns=stocks)
for asset in stocks:
    am = arch_model(log_returns_forward_full[asset] * 100, vol='Garch', p=1, q=1, dist='Normal')
    res = am.fit(disp='off')
    garch_volatility_forward[asset] = res.conditional_volatility / 100
feat_1_f = rolling_returns_forward.add_suffix('_roll_ret')
feat_2_f = rolling_vol_forward.add_suffix('_roll_vol')
feat_3_f = garch_volatility_forward.add_suffix('_garch_vol')

features_forward_full = pd.concat([feat_1_f, feat_2_f, feat_3_f], axis=1).dropna()
features_forward = features_forward_full.loc[FORWARD_START:FORWARD_END]
log_returns_forward = log_returns_forward_full.loc[features_forward.index]

print(f"Forward Test feature matrix shape: {features_forward.shape}")

In [ ]:
# Forward test

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np
import yfinance as yf

print("Running forward test...")
forward_env = PortfolioEnv(features_forward, log_returns_forward, initial_capital=1.0, rebalance_frequency=REBALANCE_FREQUENCY, transaction_cost_rate=TRANSACTION_COST_RATE)
model = SAC.load("sac_portfolio_allocator")

forward_weights_history = []
forward_returns_history = []

obs, _ = forward_env.reset()
terminated = False

while not terminated:
    action, _states = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = forward_env.step(action)

    forward_weights_history.append(forward_env.portfolio_weights)
    forward_returns_history.append(info['portfolio_return'])
forward_dates = log_returns_forward.index[:-1]

forward_weights_df = pd.DataFrame(forward_weights_history, columns=stocks, index=forward_dates)
forward_daily_returns = pd.Series(forward_returns_history, index=forward_dates, name="Forward_Agent")
bench_data_f = yf.download(BENCHMARK_TICKERS, start=FETCH_START_FORWARD, end=FORWARD_END, progress=False)["Close"]
bench_data_f = bench_data_f.ffill().dropna()

bench_pct_returns_f = bench_data_f.pct_change().dropna()
bench_simple_portfolio_ret_f = (bench_pct_returns_f * BENCHMARK_WEIGHTS).sum(axis=1)
bench_daily_ret_full = np.log(1 + bench_simple_portfolio_ret_f)
common_forward_dates = forward_dates.intersection(bench_daily_ret_full.index)
forward_daily_returns = forward_daily_returns.loc[common_forward_dates]
bench_daily_ret_f = bench_daily_ret_full.loc[common_forward_dates]
forward_portfolio_cum = np.exp(forward_daily_returns.cumsum())
forward_perf = pd.concat([perf_stats(forward_daily_returns), perf_stats(bench_daily_ret_f)], axis=1)
forward_perf.columns = ["Forward RL Agent", BENCHMARK_NAME]
print("\n--- Forward-test results ---")
display(forward_perf)
fig_forward = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.05,
    row_heights=[0.7, 0.3]
)
fig_forward.add_trace(go.Scatter(
    x=common_forward_dates,
    y=forward_portfolio_cum,
    mode='lines+markers',
    name='Forward RL Agent',
    line=dict(color='#006400', width=2.5),
    marker=dict(size=6)
), row=1, col=1)
fig_forward.add_trace(go.Scatter(
    x=common_forward_dates,
    y=np.exp(bench_daily_ret_f.cumsum()),
    mode='lines',
    name=BENCHMARK_NAME,
    line=dict(color='#7f7f7f', width=2, dash='dash')
), row=1, col=1)
fig_forward.add_trace(go.Scatter(
    x=common_forward_dates,
    y=bench_daily_ret_f,
    mode='lines',
    name='Benchmark Daily Return',
    line=dict(color='#1f77b4', width=1)
), row=2, col=1)

fig_forward.update_layout(
    title='Forward-test performance',
    xaxis_title='Date',
    template='plotly_white',
    hovermode='x unified',
    height=750,
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01, bordercolor="gray", borderwidth=1)
)
fig_forward.update_yaxes(title_text="Growth of ₹1", row=1, col=1)
fig_forward.update_yaxes(title_text="Daily Return", tickformat=".1%", row=2, col=1)

fig_forward.show()

In [ ]:
# Forward-test risk

print("Calculating forward-test risk metrics...")
cov_matrix_f = np.cov(forward_daily_returns, bench_daily_ret_f)
forward_beta = cov_matrix_f[0, 1] / cov_matrix_f[1, 1]
ann_ret_agent_f = forward_perf.loc["Annual Return", "Forward RL Agent"]
ann_ret_bench_f = forward_perf.loc["Annual Return", BENCHMARK_NAME]
forward_jensens_alpha = ann_ret_agent_f - (RISK_FREE_RATE + forward_beta * (ann_ret_bench_f - RISK_FREE_RATE))
forward_active_returns = forward_daily_returns - bench_daily_ret_f
forward_tracking_error = forward_active_returns.std() * np.sqrt(252)
forward_active_return_ann = ann_ret_agent_f - ann_ret_bench_f
forward_information_ratio = forward_active_return_ann / forward_tracking_error if forward_tracking_error != 0 else np.nan
forward_advanced_stats = pd.Series({
    "Portfolio Beta": forward_beta,
    "Jensen's Alpha": forward_jensens_alpha,
    "Tracking Error": forward_tracking_error,
    "Information Ratio": forward_information_ratio
})

print("\n--- Forward-test risk metrics ---")
for metric, val in forward_advanced_stats.items():
    if "Ratio" in metric or "Beta" in metric:
        print(f"{metric}: {val:.3f}")
    else:
        print(f"{metric}: {val:.2%}")

In [ ]:
# Regime test

import numpy as np
import pandas as pd
import plotly.graph_objects as go

print("Classifying market regimes...")
rolling_1m_bench = bench_aligned.rolling(window=21).sum()
conditions = [
    (rolling_1m_bench > 0.02),
    (rolling_1m_bench < -0.02)
]
choices = ['Bull', 'Bear']
regimes = np.select(conditions, choices, default='Chop')
regime_df = pd.DataFrame({
    'Agent_Return': drl_aligned,
    'Benchmark_Return': bench_aligned,
    'Regime': regimes
}, index=drl_aligned.index)
regime_df = regime_df.dropna()
regime_df = regime_df[regime_df['Regime'] != '0']
def calc_regime_metrics(returns_series):
    ann_ret = (np.exp(returns_series.mean() * 252) - 1)
    ann_vol = returns_series.std() * np.sqrt(252)
    sharpe = ann_ret / ann_vol if ann_vol != 0 else 0
    return pd.Series([ann_ret, ann_vol, sharpe], index=['Ann. Return', 'Ann. Volatility', 'Sharpe Ratio'])
agent_regime_stats = regime_df.groupby('Regime')['Agent_Return'].apply(calc_regime_metrics).unstack()
bench_regime_stats = regime_df.groupby('Regime')['Benchmark_Return'].apply(calc_regime_metrics).unstack()

print("\n--- Agent performance by regime ---")
display(agent_regime_stats.style.format("{:.3f}"))

print(f"\n--- {BENCHMARK_NAME} Performance by Market Regime ---")
display(bench_regime_stats.style.format("{:.3f}"))
regime_labels = agent_regime_stats.index.tolist()
agent_sharpes = agent_regime_stats['Sharpe Ratio'].tolist()
bench_sharpes = bench_regime_stats['Sharpe Ratio'].tolist()

fig_regime = go.Figure(data=[
    go.Bar(name='RL Agent', x=regime_labels, y=agent_sharpes, marker_color='#2ca02c'),
    go.Bar(name=BENCHMARK_NAME, x=regime_labels, y=bench_sharpes, marker_color='#7f7f7f')
])

fig_regime.update_layout(
    title='Sharpe ratio by market regime',
    xaxis_title='Market Regime (1-Month Trailing Horizon)',
    yaxis_title='Sharpe Ratio',
    barmode='group',
    template='plotly_white',
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01, bordercolor="gray", borderwidth=1)
)

fig_regime.show()

In [ ]:
# Allocation by category

import pandas as pd
import plotly.graph_objects as go
ASSET_TO_MACRO_CATEGORY = {
    "GOLDBEES.NS": "Commodities",
    "SILVERBEES.NS": "Commodities",
    "SETF10GILT.NS": "Fixed Income",
    "INDIGRID.NS": "Listed Real Assets",
    "MCX.NS": "Indian Equity",
    "LT.NS": "Indian Equity",
    "NTPC.NS": "Indian Equity",
    "COALINDIA.NS": "Indian Equity",
    "PIDILITIND.NS": "Indian Equity",
    "WELCORP.NS": "Indian Equity",
    "CUMMINSIND.NS": "Indian Equity",
    "MANORAMA.NS": "Indian Equity",
    "SOLARINDS.NS": "Indian Equity",
}
mean_asset_weights = drl_weights_df.mean()
asset_categories = mean_asset_weights.index.to_series().map(ASSET_TO_MACRO_CATEGORY)
unmapped_assets = asset_categories[asset_categories.isna()].index.tolist()
if unmapped_assets:
    raise ValueError(f"Missing macro categories for: {', '.join(unmapped_assets)}")

macro_category_weights = mean_asset_weights.groupby(asset_categories).sum().sort_values(ascending=False)
color_map = {
    "Indian Equity": "#988ED5",
    "Global Equity": "#777777",
    "Commodities": "#E64A19",
    "Fixed Income": "#FDD835",
    "Listed Real Assets": "#ADD8E6",
}
mapped_colors = [color_map[category] for category in macro_category_weights.index]
fig_pie = go.Figure(data=[go.Pie(
    labels=macro_category_weights.index,
    values=macro_category_weights.values,
    hole=0.0,
    textinfo='label+percent',
    textfont=dict(size=14, color='black'),
    marker=dict(colors=mapped_colors, line=dict(color='#FFFFFF', width=1.5)),
    hovertemplate='<b>%{label}</b><br>Average Allocation: %{percent}<extra></extra>'
)])
fig_pie.update_layout(
    title=dict(
        text='Portfolio Allocation by Macro Category',
        x=0.5,
        y=0.95,
        font=dict(size=22, color='black')
    ),
    showlegend=False,
    width=700,
    height=700,
    template='plotly_white',
    margin=dict(t=80, b=40, l=40, r=40)
)
fig_pie.show()

In [ ]:
# Feature importance

import numpy as np
import pandas as pd
import plotly.express as px

print("Running permutation importance...")
def evaluate_agent_fitness(test_features, test_returns):
    env = PortfolioEnv(test_features, test_returns, initial_capital=1.0, rebalance_frequency=REBALANCE_FREQUENCY, transaction_cost_rate=TRANSACTION_COST_RATE)
    obs, _ = env.reset()
    terminated = False
    returns_history = []

    while not terminated:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
        returns_history.append(info['portfolio_return'])

    daily_ret = pd.Series(returns_history)
    ann_ret = (np.exp(daily_ret.mean() * 252) - 1)
    ann_vol = daily_ret.std() * np.sqrt(252)
    sharpe = (ann_ret - RISK_FREE_RATE) / ann_vol if ann_vol != 0 else 0
    return sharpe
baseline_sharpe = evaluate_agent_fitness(features_test, log_returns_test)
print(f"Baseline Sharpe: {baseline_sharpe:.3f}")
feature_names = features_test.columns.tolist()
importance_scores = {}

for feature in feature_names:
    shuffled_features = features_test.copy()
    shuffled_features[feature] = np.random.permutation(shuffled_features[feature].values)
    shuffled_sharpe = evaluate_agent_fitness(shuffled_features, log_returns_test)
    drop_in_sharpe = baseline_sharpe - shuffled_sharpe
    importance_scores[feature] = drop_in_sharpe
importance_df = pd.DataFrame.from_dict(importance_scores, orient='index', columns=['Importance'])
importance_df = importance_df.sort_values(by='Importance', ascending=True)
top_features = importance_df.tail(15)
fig_xai = px.bar(
    top_features,
    x='Importance',
    y=top_features.index,
    orientation='h',
    title="Top 15 features by Sharpe impact",
    labels={'Importance': 'Drop in Sharpe Ratio when Signal is Scrambled', 'index': 'Feature'},
    color='Importance',
    color_continuous_scale='Reds'
)

fig_xai.update_layout(
    template='plotly_white',
    height=600,
    yaxis=dict(title=""),
    xaxis=dict(title="Sharpe Ratio Impact (Higher = More impact)")
)

fig_xai.show()

In [ ]:
from google.colab import files